In [1]:
# Import required libraries
# numpy: used for numerical computations and handling arrays
# pandas: used for data manipulation and reading CSV files (e.g., pd.read_csv)

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# The os module helps interact with the operating system.
# In Kaggle notebooks, input datasets are usually stored in '/kaggle/input'.

import os

# Walk through the '/kaggle/input' directory and print all available files.
# This helps verify that the dataset has been correctly loaded into the environment
# and allows us to see the exact file paths for reading the data.
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv


In [2]:
# Import required libraries for model training and fine-tuning
import torch
# PyTorch: deep learning framework used for loading and training neural network models
import re
# re: Python's regular expression library used for text cleaning and pattern matching
import unicodedata
# unicodedata: helps normalize text (useful for cleaning special characters and Unicode text)

from datasets import Dataset
# Hugging Face Datasets: used to convert pandas data into a dataset format compatible with transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
# transformers:
# AutoTokenizer -> automatically loads the tokenizer for the selected LLM
# AutoModelForCausalLM -> loads a causal language model (e.g., Gemma, LLaMA, GPT-style models)
# TrainingArguments -> defines training configuration (batch size, epochs, learning rate, etc.)
# Trainer -> high-level API for training and evaluating transformer models

from peft import LoraConfig, get_peft_model
# PEFT (Parameter Efficient Fine-Tuning):

In [3]:
# Load the training and test datasets from the Kaggle competition input directory

# The training dataset contains labeled examples (text and corresponding emotion labels)
# that will be used to fine-tune the language model.

train = pd.read_csv("/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv")

# The test dataset contains only the input text without labels.
# After training the model, we will generate predictions for this dataset
# and create the final submission file.

test = pd.read_csv("/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv")

In [4]:
# Text preprocessing function
# This function cleans and normalizes text before it is used for training or inference.
# Cleaning the text helps improve model performance by removing unnecessary noise.

def preprocess(text):

    text = unicodedata.normalize("NFKC", str(text))

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Apply preprocessing to the sentence column of the training dataset
train["sentence"] = train["sentence"].apply(preprocess)

# Apply the same preprocessing to the test dataset
test["sentence"] = test["sentence"].apply(preprocess)

In [5]:
# Function to create an instruction-style prompt for the language model.
# The prompt follows the Instruction → Input → Response format commonly
# used by instruction-tuned LLMs (such as Gemma, LLaMA, etc.).

# During training:
#   The label is appended to the prompt so the model learns the correct response.

# During inference:
#   The label is not provided, and the model generates the predicted emotion.

def create_prompt(text,language=None,label=None):

    prompt = f"""### Instruction:
                 Classify the emotion expressed in the text.
                 The text is written in {language}
                 ### Input:
                 {text}

                 ### Response:
             """
    if label is not None:
        prompt += f" {label}"

    return prompt

In [6]:
# Create a new column called "prompt" in the training dataset.
# Each row is converted into an instruction-style prompt using the create_prompt() function.

# The lambda function applies create_prompt() to every row:
#   - x["sentence"] provides the input text
#   - x["language"] provides the input text language
#   - x["label"] provides the correct emotion label

# The resulting prompt will be used as the training input for the language model.

train["prompt"] = train.apply(
    lambda x: create_prompt(x["sentence"],x['language'], x["label"]),
    axis=1
)

In [7]:
# Convert the pandas DataFrame into a Hugging Face Dataset object.
# The Trainer API from the transformers library works efficiently with
# Hugging Face datasets for tasks like tokenization, batching, and training.

# This step allows us to use dataset.map() for preprocessing and
# easily feed the data into the model training pipeline.

dataset = Dataset.from_pandas(train)

In [ ]:
# Install required libraries for transformer models and efficient training
# transformers -> Hugging Face library for loading and training LLMs
# accelerate -> helps optimize training across devices (CPU/GPU)
# bitsandbytes -> enables memory-efficient quantization (useful for large models)
# triton -> used internally for optimized GPU kernels

!pip install -U transformers accelerate bitsandbytes triton

# Login to Hugging Face Hub
# This is required if the model or resources require authentication.
# Avoid hardcoding tokens in notebooks for security reasons.

from huggingface_hub import login
login("hf_token")


# Specify the pretrained model name
# Here we use Gemma 3 (1B parameters) instruction-tuned model
model_name = "google/gemma-3-1b-it"

# Load the tokenizer associated with the model
# The tokenizer converts text into tokens that the model can process
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

# Load the pretrained causal language model
# device_map="auto" automatically places the model on available GPU/CPU
# torch_dtype="auto" selects the optimal precision supported by the hardware
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype="auto"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [9]:
# Configure LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.
# Instead of updating all model parameters, LoRA adds small trainable adapters
# to specific layers of the model. This greatly reduces memory usage and
# training time while maintaining good performance.

config = LoraConfig(
    r=8,
    # Rank of the LoRA update matrices (controls the size of the adapters)
    lora_alpha=16,
    # Scaling factor that determines how strongly the LoRA updates affect the model
    target_modules=["q_proj", "v_proj"],
    # Apply LoRA to the query and value projection layers of the attention mechanism
    lora_dropout=0.1,
    # Dropout applied to LoRA layers to reduce overfitting
    bias="none"
    # Specifies whether bias parameters are trained (here they are not)
)
# Attach the LoRA adapters to the base model
# After this step, only the LoRA parameters will be trained
model = get_peft_model(model, config)

In [10]:
# Tokenization function
# This function converts text prompts into token IDs that the model can understand.
# The tokenizer splits the text into tokens and maps them to numerical IDs.
def tokenize(example):

    tokens = tokenizer(
        example["prompt"], # Instruction-style prompt created earlier
        truncation=True,   # Truncate sequences longer than max_length
        padding="max_length", # Pad shorter sequences to the same length
        max_length=256  # Maximum sequence length for the model input
    )

    # For causal language models, the labels are usually the same as input_ids.
    # This allows the model to learn to predict the next token in the sequence.
    tokens["labels"] = tokens["input_ids"]
    
    return tokens

# Apply the tokenization function to the entire dataset
# batched=True processes multiple rows at once for faster execution
# remove_columns removes original columns since only tokenized data is needed

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names
)

# Convert dataset format to PyTorch tensors so it can be used by the Trainer
dataset.set_format("torch")

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [11]:
# Define training configuration for the model using Hugging Face TrainingArguments.
# These parameters control how the model is trained, optimized, and logged.

training_args = TrainingArguments(
    output_dir="./emotion_model",
    # Directory where the trained model checkpoints and logs will be saved
    per_device_train_batch_size=4,
    # Number of training samples processed per GPU/CPU at a time
    gradient_accumulation_steps=4,
    # Accumulates gradients over multiple steps to simulate a larger batch size
    # Effective batch size = 4 × 4 = 16
    num_train_epochs=4,
    # Number of complete passes through the training dataset
    learning_rate=2e-4,
    # Initial learning rate used by the optimizer
    logging_steps=20,
    # Frequency (in steps) for logging training metrics
    warmup_steps=60,
    # Gradually increases the learning rate during the initial steps
    # to stabilize training
    save_strategy="epoch",
    # Saves model checkpoints at the end of each training epoch
    lr_scheduler_type="cosine",
    # Uses cosine learning rate decay during training
    weight_decay=0.01,
    # Regularization technique to prevent overfitting
    report_to="tensorboard",
    # Sends training logs to TensorBoard for visualization
    fp16=True
    # Enables mixed precision (16-bit floating point) to reduce GPU memory usage
    # and speed up training on supported hardware
)

In [12]:
# Function to generate emotion predictions using the fine-tuned model.
# The function takes a text input, converts it into the same prompt format
# used during training, and then asks the model to generate the predicted label.
def predict(text):
    
    # Create the instruction-style prompt without the label
    prompt = create_prompt(text)

    # Tokenize the prompt and move tensors to the same device as the model (CPU/GPU)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate model output
    # max_new_tokens limits the number of tokens the model can generate
    # do_sample=False ensures deterministic output (no randomness)
    output = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    # Decode generated tokens back into text
    result = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract the predicted emotion from the generated text
    # The model output appears after "### Response:"
    emotion = result.split("### Response:")[-1].strip().split("\n")[0]

    # Convert textual prediction to numeric label format expected by submission
    if "Positive" in emotion:
        emotion = "1"
    elif "Negative" in emotion:
        emotion = "0"
    else:
        emotion = "1"
        
    return emotion

In [13]:
# Initialize the Hugging Face Trainer
# The Trainer API simplifies the training process by handling
# batching, gradient updates, logging, and checkpointing.

trainer = Trainer(
    model=model,
    # The model to be fine-tuned (Gemma with LoRA adapters applied)
    args=training_args,
    # Training configuration defined earlier (batch size, epochs, learning rate, etc.)
    train_dataset=dataset,
    # Tokenized training dataset used to fine-tune the model
)

# Start the training process
# This will run the training loop according to the parameters
# defined in TrainingArguments (epochs, learning rate schedule, etc.)
trainer.train()

2026-03-08 02:26:09.793188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772936769.965914      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772936770.019454      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772936770.446691      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772936770.446718      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772936770.446720      24 computation_placer.cc:177] computation placer alr

Step,Training Loss
20,15.780173
40,15.204680
60,14.981302
80,14.902353
100,14.691809
120,14.753638
140,14.760913
160,14.789288
180,14.772443
200,14.740755


TrainOutput(global_step=228, training_loss=14.909169113426877, metrics={'train_runtime': 781.1962, 'train_samples_per_second': 4.608, 'train_steps_per_second': 0.292, 'total_flos': 3863208237465600.0, 'train_loss': 14.909169113426877, 'epoch': 4.0})

In [14]:
# Generate predictions for the test dataset.
# We iterate through each row of the test DataFrame, extract the sentence,
# and use the trained model to predict the corresponding emotion label.
predictions = []

for _, row in test.iterrows():
    text = row["sentence"] # Input text for which we want to predict the emotion
    id = row["ID"] # Unique ID used for the competition submission

    # Use the prediction function to generate the emotion label
    emotion = predict(text)
    
    # Store the prediction along with the ID in a list of dictionaries
    predictions.append({
        "ID": id,
        "label": emotion
    })

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [15]:
# Convert the list of prediction dictionaries into a pandas DataFrame.
# Each entry contains the test sample ID and the predicted emotion label.
df = pd.DataFrame(predictions)

# Save the DataFrame as a CSV file for competition submission.
# index=False ensures that pandas does not add an extra index column.
df.to_csv("submission.csv", index=False)

In [16]:
# Remove the saved model directory to reduce the notebook output size.
# During training, the Trainer saves model checkpoints in the folder
# specified by `output_dir` (here: ./emotion_model).
#
# For competition submissions (e.g., Kaggle), large saved model files
# are not required after generating the submission.csv file and may
# exceed notebook size limits. This step deletes the directory to
# keep the notebook lightweight.
import shutil

shutil.rmtree("./emotion_model")